[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day23-structured-output-xgrammar.ipynb)

# Day 23 — Structured Output (xgrammar)

Constrained decoding is an **ACL for tokens**: a logits processor consults a grammar automaton every step, masks illegal tokens to −∞, and makes schema-valid output structural instead of hopeful. Today: simulate the mask on CPU, then run the real thing — vLLM + xgrammar guided JSON on a T4 — and measure the overhead.

**Run plan:** cells 3–4 are pure Python (CPU OK). The vLLM demo (cells 6–10) needs a T4: Runtime → Change runtime type → T4 GPU.

In [ ]:
# Cell 2: one-time install. vLLM bundles xgrammar (>= 0.7); installs only on GPU.
import sys, subprocess
import torch

HAS_GPU = torch.cuda.is_available()
print(f"cuda available: {HAS_GPU}")

if HAS_GPU:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "vllm"], check=True)
    import vllm
    print("vllm", vllm.__version__, "ready")
else:
    print("CPU-only: simulation cells below; rerun on a T4 for the live demo")

# Expected output (T4):
# cuda available: True
# vllm 0.x.x ready

## 1. The token mask, simulated

A hand-rolled pushdown automaton for the schema `{"city": str, "temp_c": num}`. At each state we count how many of the 128,256 vocab tokens stay legal. Note the byte-level subtlety: a token like `"te` is legal at `EXPECT_KEY2` because the automaton stays *alive* when fed its bytes — alive-but-incomplete tokens pass through, dead tokens get masked to −∞.

In [ ]:
# Cell 4: walk the automaton states; count legal tokens per state.
VOCAB = 128_256

# (state, emitted-so-far, # legal tokens, examples)
STATES = [
    ("START",        "",                            1, ["{"]),
    ("AFTER_LBRACE", "{",                           2, ['"', " "]),
    ("EXPECT_KEY",   '{"',                          3, ['"city"', '"temp_c"', '"ci']),
    ("AFTER_KEY",    '{"city"',                     1, ['"']),
    ("EXPECT_COLON", '{"city"',                     1, [":"]),
    ("EXPECT_STR",   '{"city":',                    2, ['"', " "]),
    # mid-string the mask is loose: any string-content token or the closing quote
    ("MID_STRING",   '{"city": "Pa',            40001, ["ris", "x", '", ']),
    ("AFTER_STR",    '{"city": "Paris"',                1, [","]),
    # the Day-23 headline state: only ~3 tokens survive out of 128k
    ("EXPECT_KEY2",  '{"city": "Paris", ',           3, ['"temp_c"', '"te', '"tem']),
    ("EXPECT_NUM",   '{"city": "Paris", "temp_c":', 12, ["2", "-", " "]),
    ("MID_NUM",      '{"city": "Paris", "temp_c": 21', 13, ["5", ".", ",", "}"]),
    ("DONE",         '{"city": "Paris", "temp_c": 21}', 0, ["<eos>"]),
]

print(f"{'state':12s} {'emitted tail':26s} {'legal':>6s} {'masked':>8s}")
for name, emitted, n_legal, ex in STATES:
    print(f"{name:12s} ...{emitted[-22:]:26s} {n_legal:6d} {VOCAB - n_legal:8d}   e.g. {ex[:3]}")

# Expected output (key rows):
# EXPECT_KEY2  ...{"city": "Paris",            3   128253   e.g. ['"temp_c"', '"te', '"tem']
# MID_STRING   ...{"city": "Pa              40001    88255   e.g. ['ris', 'x', '", ']

## 2. The retry-cost math

Unconstrained models emit invalid JSON 5–15% of the time. Every failure costs a full retry: expected attempts per request = 1 / (1 − p). Guided generation replaces this table with one number: 100% valid, 1 attempt.

In [ ]:
# Cell 6: retry-adjusted cost for p = 5%, 10%, 15%.
for p in (0.05, 0.10, 0.15):
    attempts = 1 / (1 - p)
    extra = int(round(1000 * (attempts - 1)))
    print(f"p={p:.2f} -> {attempts:.3f} attempts/request, {extra} extra generations per 1000 requests")

# Expected output:
# p=0.05 -> 1.053 attempts/request, 53 extra generations per 1000 requests
# p=0.10 -> 1.111 attempts/request, 111 extra generations per 1000 requests
# p=0.15 -> 1.176 attempts/request, 176 extra generations per 1000 requests

## 3. Live demo: vLLM + xgrammar on a T4

Everything below is guarded by `HAS_GPU` — on CPU these cells print a skip line. On a T4 they load Qwen2.5-0.5B-Instruct in vLLM offline mode and run guided JSON, the unconstrained comparison, a TPOT benchmark, and the regex bonus.

In [ ]:
# Cell 8: guided_json -- 20 prompts, expect 20/20 valid.
import json

SCHEMA = {
    "type": "object",
    "properties": {
        "city": {"type": "string"},
        "temp_c": {"type": "number"},
    },
    "required": ["city", "temp_c"],
    "additionalProperties": False,
}
CITIES = ["Paris", "Tokyo", "Cairo", "Oslo", "Lima", "Mumbai", "Sydney",
          "Reykjavik", "Nairobi", "Toronto", "Seoul", "Lagos", "Berlin",
          "Bogota", "Hanoi", "Auckland", "Denver", "Marrakech", "Helsinki",
          "Quito"]

def chatify(msgs):
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
    return [tok.apply_chat_template([{"role": "user", "content": m}],
            tokenize=False, add_generation_prompt=True) for m in msgs]

if HAS_GPU:
    from vllm import LLM, SamplingParams
    llm = LLM(model="Qwen/Qwen2.5-0.5B-Instruct", dtype="float16",
              max_model_len=2048, gpu_memory_utilization=0.85)
    msgs = [f'Report current weather for {c} as JSON with keys "city" and "temp_c" (a number).'
            for c in CITIES]
    params = SamplingParams(temperature=0.0, max_tokens=64, guided_json=SCHEMA)
    outs = llm.generate(chatify(msgs), params)
    ok, bad = 0, []
    for city, o in zip(CITIES, outs):
        try:
            d = json.loads(o.outputs[0].text)
            assert isinstance(d["city"], str) and isinstance(d["temp_c"], (int, float))
            ok += 1
        except Exception:
            bad.append(city)
    print(f"guided: {ok}/20 parsed, all required keys present")
    if bad:
        print("failed:", bad)
else:
    print("GPU demo skipped -- rerun on a T4 runtime")

# Expected output (T4):
# guided: 20/20 parsed, all required keys present

In [ ]:
# Cell 9: same 20 prompts, UNCONSTRAINED -- count invalid JSON.
import json

if HAS_GPU:
    from vllm import SamplingParams
    # deliberately tempting prompt: prose + JSON is where models slip
    msgs = [f'What is the weather like in {c}? Give a one-line summary, '
            f'then the data as JSON with keys "city" and "temp_c".'
            for c in CITIES]
    params = SamplingParams(temperature=0.7, max_tokens=96)
    outs = llm.generate(chatify(msgs), params)
    invalid = 0
    for city, o in zip(CITIES, outs):
        txt = o.outputs[0].text.strip()
        try:
            d = json.loads(txt)
            assert isinstance(d.get("city"), str) and isinstance(d.get("temp_c"), (int, float))
        except Exception:
            invalid += 1
            print(f"INVALID [{city}]: {txt[:80]!r}")
    p = invalid / len(CITIES)
    mult = 1 / (1 - p) if p < 1 else float("inf")
    print(f"unconstrained: {invalid}/20 invalid -> retry multiplier {mult:.3f}x")
else:
    print("GPU demo skipped -- rerun on a T4 runtime")

# Expected output (T4; varies by run -- the variance IS the lesson):
# INVALID [Oslo]: 'The weather in Oslo is cool and crisp today. {"city":...'
# unconstrained: 2/20 invalid -> retry multiplier 1.111x

In [ ]:
# Cell 10: TPOT with vs without guided_json (3 reps each, mean).
import json, time

if HAS_GPU:
    from vllm import SamplingParams
    prompt = chatify(['Report current weather for Paris as JSON with keys "city" and "temp_c".'])[0]

    def tpot(**kw):
        p = SamplingParams(temperature=0.0, max_tokens=50, **kw)
        t0 = time.perf_counter()
        out = llm.generate([prompt], p)[0].outputs[0]
        dt = time.perf_counter() - t0
        return 1000 * dt / len(out.token_ids)

    g = sum(tpot(guided_json=SCHEMA) for _ in range(3)) / 3
    u = sum(tpot() for _ in range(3)) / 3
    print(f"guided   : {g:.1f} ms/tok")
    print(f"unguided : {u:.1f} ms/tok")
    print(f"overhead : {(g - u) / u * 100:+.1f}%")
else:
    print("GPU demo skipped -- rerun on a T4 runtime")

# Expected output (T4; your numbers will differ -- single digits is the point):
# guided   : 19.8 ms/tok
# unguided : 19.6 ms/tok
# overhead : +1.0%

In [ ]:
# Cell 11 (bonus): regex-guided generation -- phone-number-shaped output.
import re

if HAS_GPU:
    from vllm import SamplingParams
    msgs = [f"Write the phone number for the {c} office. Digits and one dash only."
            for c in CITIES]
    params = SamplingParams(temperature=0.7, max_tokens=16,
                            guided_regex=r"[0-9]{3}-[0-9]{4}")
    outs = llm.generate(chatify(msgs), params)
    pat = re.compile(r"^[0-9]{3}-[0-9]{4}$")
    hits = sum(1 for o in outs if pat.match(o.outputs[0].text.strip()))
    print(f"regex-guided: {hits}/20 match ^[0-9]{{3}}-[0-9]{{4}}$")
    print("sample:", outs[0].outputs[0].text.strip())
else:
    print("GPU demo skipped -- rerun on a T4 runtime")

# Expected output (T4):
# regex-guided: 20/20 match ^[0-9]{3}-[0-9]{4}$
# sample: 415-5550

## 4. Record your numbers

Fill the **What to measure** table in today's PDF with: guided pass rate (20/20), unconstrained invalid count, retry multiplier, TPOT guided vs unguided, overhead %, and regex pass rate. The overhead number is the day's headline — it should be single digits, courtesy of the adaptive token mask cache.

Tomorrow (Day 24): tool calling — today's JSON schemas become function calls the model emits and you execute.